In [ ]:
import timm
import torch.nn as nn
import copy

NUM_CLASSES = 4
class ViTSegmentation(nn.Module):

    def __init__(self):
        super().__init__()
        self.vit = timm.create_model(
            "vit_tiny_patch16_224",
            pretrained=True,
            in_chans=6,
            num_classes=0
        )
        embed_dim = self.vit.embed_dim
        self.decoder = nn.Sequential(
            nn.Conv2d(embed_dim,128,1),
            nn.ReLU(),
            nn.Conv2d(128,64,3,padding=1),
            nn.ReLU(),
            nn.Conv2d(64,NUM_CLASSES,1)
        )

    def forward(self,x):
        B,C,H,W = x.shape
        tokens = self.vit.patch_embed(x)
        if tokens.dim()==4:
            B,C,h,w = tokens.shape
            tokens = tokens.flatten(2).transpose(1,2)

        cls = self.vit.cls_token.expand(B,-1,-1)
        tokens = torch.cat([cls,tokens],dim=1)

        tokens = tokens + self.vit.pos_embed[:,:tokens.shape[1]]
        tokens = self.vit.pos_drop(tokens)
        for blk in self.vit.blocks:
            tokens = blk(tokens)

        tokens = self.vit.norm(tokens)
        tokens = tokens[:,1:]  

        N = tokens.shape[1]
        S = int(N**0.5)
        
        feat = tokens.transpose(1,2).reshape(B,self.vit.embed_dim,S,S)
        out = self.decoder(feat)
        out = nn.functional.interpolate(out,size=(H,W),mode="bilinear")

        return out

c:\Users\Asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class SSLTeacherStudent(nn.Module):
    def __init__(self):
        super().__init__()
        self.student = ViTSegmentation()
        self.teacher = copy.deepcopy(self.student)
        for p in self.teacher.parameters():
            p.requires_grad = False

    def forward_student(self,x):
        return self.student(x)

    def forward_teacher(self,x):
        with torch.no_grad():
            return self.teacher(x)

In [3]:
def update_teacher(student_model, teacher_model, alpha=0.99):

    for teacher_param, student_param in zip(
        teacher_model.parameters(),
        student_model.parameters()
    ):

        teacher_param.data = (
            alpha * teacher_param.data
            + (1-alpha) * student_param.data
        )

augmentation

In [ ]:
import torchvision.transforms as T
import random

IMG_SIZE = (224,224)
weak_aug = T.Compose([
    T.Resize(IMG_SIZE),
    T.RandomHorizontalFlip(p=0.5),
    T.ToTensor()
])

strong_aug = T.Compose([
    T.Resize(IMG_SIZE),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(0.4,0.4,0.4,0.1),
    T.RandomRotation(10),
    T.GaussianBlur(3),
    T.ToTensor()
])

In [5]:
def generate_pseudo_labels(logits, threshold=0.7):

    probs = torch.softmax(logits, dim=1)

    max_probs, pseudo_labels = torch.max(probs, dim=1)

    mask = max_probs.ge(threshold).float()

    return pseudo_labels, mask

In [ ]:
model = SSLTeacherStudent().to("cuda")

optimizer = torch.optim.Adam(model.student.parameters(), lr=1e-4)

criterion = nn.CrossEntropyLoss()

for (x_l, y_l), x_u in zip(labeled_loader, unlabeled_loader):

    x_l = x_l.cuda()
    y_l = y_l.cuda()
    x_u = x_u.cuda()

    pred_l = model.forward_student(x_l)
    sup_loss = criterion(pred_l, y_l)

    with torch.no_grad():
        teacher_pred = model.forward_teacher(x_u)
        pseudo = torch.argmax(teacher_pred, dim=1)

    student_pred = model.forward_student(x_u)

    unsup_loss = criterion(student_pred, pseudo)

    loss = sup_loss + 0.5 * unsup_loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    update_teacher(model.student, model.teacher, alpha=0.99)

In [2]:
import streamlit as st
from PIL import Image, ImageDraw, ImageFont
import numpy as np
import io
import os
import math
from collections import Counter
from scipy import ndimage
import cv2  # Added for Grad-CAM heatmap processing

import torch
import torch.nn as nn
import torchvision.transforms.functional as TF
import timm
import torch.nn.functional as F

IMG_SIZE = (384, 384)           
NUM_CLASSES = 4
CLASS_NAMES = {0: "no-damage", 1: "minor", 2: "major", 3: "destroyed"}
DEFAULT_CHECKPOINT = r"C:\Users\Asus\Downloads\EARTHQUAKE-TURKEY\EARTHQUAKE-TURKEY\checkpoints_4class\vit_tiny_epoch03_miou0.6246.pth"  

st.set_page_config(page_title="DamageSeg Inference", layout="centered", initial_sidebar_state="expanded")
st.title("📡 Building Damage Segmentation — Inference")
st.markdown("Upload matching pre/post images. Model expects pre+post stacked (6 channels) and images are resized to the training input size.")

# ---------------- Grad-CAM Class ----------------
class ViTSegmentationGradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None
        self.hook_handles = []
        
        # Attach hooks to the target ViT block
        self.hook_handles.append(target_layer.register_forward_hook(self.save_activation))
        self.hook_handles.append(target_layer.register_backward_hook(self.save_gradient))

    def save_activation(self, module, input, output):
        self.activations = output

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def process_vit_tokens(self, tensor):
        # Drop the [CLS] token at index 0
        tokens = tensor[:, 1:, :]
        b, n, c = tokens.shape
        h = w = int(n ** 0.5)
        # Reshape to (Batch, Channels, Height, Width)
        spatial_map = tokens.transpose(1, 2).reshape(b, c, h, w)
        return spatial_map

    def generate_heatmap(self, input_tensor, target_class):
        self.model.eval()
        self.model.zero_grad()
        
        # 1. Forward Pass
        logits = self.model(input_tensor)
        
        # 2. Get the prediction mask
        pred_mask = logits.argmax(dim=1)
        class_mask = (pred_mask == target_class).float()
        
        # 3. Compute score for backprop
        if class_mask.sum() == 0:
            score = logits[0, target_class, :, :].sum()
        else:
            score = (logits[0, target_class, :, :] * class_mask[0]).sum()
            
        # 4. Backward Pass (Compute gradients)
        score.backward(retain_graph=True)
        
        # 5. Process ViT tokens into 2D spatial maps
        acts = self.process_vit_tokens(self.activations)[0].cpu().data.numpy()
        grads = self.process_vit_tokens(self.gradients)[0].cpu().data.numpy()
        
        # 6. Global Average Pooling on gradients
        weights = np.mean(grads, axis=(1, 2))
        
        # 7. Weighted combination of activations
        cam = np.zeros(acts.shape[1:], dtype=np.float32)
        for i, w in enumerate(weights):
            cam += w * acts[i, :, :]
            
        # 8. ReLU and Normalize
        cam = np.maximum(cam, 0)
        cam = cv2.resize(cam, (input_tensor.shape[3], input_tensor.shape[2]))
        cam = cam - np.min(cam)
        cam_max = np.max(cam)
        if cam_max != 0:
            cam = cam / cam_max
            
        return cam

    def remove_hooks(self):
        for handle in self.hook_handles:
            handle.remove()

# ---------------- Utilities ----------------
def make_device():
    return "cuda" if torch.cuda.is_available() else "cpu"

DEVICE = make_device()
st.sidebar.write(f"Device: **{DEVICE}**")

def preprocess_pil(pil_img, target_size=IMG_SIZE):
    img = pil_img.convert("RGB")
    img = img.resize(target_size, resample=Image.BILINEAR)
    arr = np.array(img).astype(np.float32) / 255.0
    return arr

def tensor_from_pair(pre_arr, post_arr, device):
    pre_pil = Image.fromarray((pre_arr * 255).astype(np.uint8))
    post_pil = Image.fromarray((post_arr * 255).astype(np.uint8))
    pre_t = TF.to_tensor(pre_pil)
    post_t = TF.to_tensor(post_pil)
    inp = torch.cat([pre_t, post_t], dim=0).unsqueeze(0).to(device)
    return inp

def overlay_prediction_on_post(post_arr_uint8, pred_mask):
    color_map = {
        1: (255,165,0,140),
        2: (255,69,0,160),
        3: (200,0,0,180)
    }
    H, W = pred_mask.shape
    overlay = np.zeros((H, W, 4), dtype=np.uint8)
    for c, col in color_map.items():
        mask = (pred_mask == c)
        if mask.any():
            overlay[mask] = col 
    post_rgba = np.dstack([post_arr_uint8, np.full((H,W),255,dtype=np.uint8)])
    alpha = overlay[...,3:4].astype(np.float32) / 255.0
    out_rgb = (post_rgba[...,:3].astype(np.float32) * (1-alpha) + overlay[...,:3].astype(np.float32) * alpha).astype(np.uint8)
    return Image.fromarray(out_rgb).convert("RGBA")

def per_building_counts(pred_mask, min_area=8, treat_empty_as_no_damage=True):
    source = (pred_mask > 0).astype(np.uint8)
    labeled, ncomp = ndimage.label(source)
    counts = Counter()
    comps = []
    for comp in range(1, ncomp+1):
        comp_mask = (labeled == comp)
        area = int(comp_mask.sum())
        if area < min_area:
            continue
        vals, cnts = np.unique(pred_mask[comp_mask], return_counts=True)
        mode = int(vals[np.argmax(cnts)])
        counts[mode] += 1
        comps.append((comp, mode, area))
    if sum(counts.values()) == 0 and treat_empty_as_no_damage:
        counts[0] = 1
    return counts, comps

def draw_text_on_image(pil_img, counter, font_size=14, pad=10):
    img = pil_img.convert("RGBA")
    W, H = img.size
    text = " | ".join([f"{CLASS_NAMES[c]}: {counter.get(c,0)}" for c in range(NUM_CLASSES)])
    try:
        font = ImageFont.truetype("arial.ttf", font_size)
    except:
        font = ImageFont.load_default()
    draw_tmp = ImageDraw.Draw(img)
    max_width = W - 2*pad
    words = text.split(" ")
    wrapped = ""
    line = ""
    for word in words:
        test_line = (line + " " + word).strip()
        try:
            bbox = draw_tmp.textbbox((0,0), test_line, font=font)
            w = bbox[2] - bbox[0]
        except:
            w, _ = draw_tmp.textsize(test_line, font=font)
        if w <= max_width:
            line = test_line
        else:
            wrapped += line + "\n"
            line = word
    wrapped += line
    try:
        bbox = draw_tmp.multiline_textbbox((0,0), wrapped, font=font)
        text_w = bbox[2] - bbox[0]; text_h = bbox[3] - bbox[1]
    except:
        text_w, text_h = draw_tmp.textsize(wrapped, font=font)
    banner_h = text_h + 2*pad
    new_img = Image.new("RGBA", (W, H + banner_h), (0,0,0,0))
    new_img.paste(img, (0, banner_h))
    draw = ImageDraw.Draw(new_img)
    draw.rectangle([0, 0, W, banner_h], fill=(0,0,0,150))
    x = (W - text_w) // 2
    y = pad
    draw.multiline_text((x, y), wrapped, font=font, fill=(255,255,255,255), align="center")
    return new_img.convert("RGB")

def resize_vit_pos_embed_for_input(model, img_size):
    try:
        vit = model.vit
        if not hasattr(vit, "pos_embed"):
            return
        pos = vit.pos_embed
        patch_embed = vit.patch_embed
        if hasattr(patch_embed, "kernel_size"):
            if isinstance(patch_embed.kernel_size, tuple):
                patch_size = patch_embed.kernel_size[0]
            else:
                patch_size = patch_embed.kernel_size
        else:
            patch_size = 16
        ph, pw = img_size[0] // patch_size, img_size[1] // patch_size
        new_grid = ph * pw
        old_grid = pos.shape[1] - 1
        if new_grid == old_grid:
            return
        cls_tok = pos[:, :1, :]
        grid_tok = pos[:, 1:, :].transpose(1,2).reshape(1, pos.shape[2], int(math.sqrt(old_grid)), int(math.sqrt(old_grid)))
        grid_tok_resized = F.interpolate(grid_tok, size=(ph, pw), mode='bilinear', align_corners=False)
        new_grid_tok = grid_tok_resized.reshape(1, pos.shape[2], ph*pw).transpose(1,2)
        new_pos = torch.cat([cls_tok, new_grid_tok], dim=1).to(pos.device)
        vit.pos_embed = nn.Parameter(new_pos)
        print(f"Resized pos_embed: {int(math.sqrt(old_grid))}x{int(math.sqrt(old_grid))} -> {ph}x{pw} (tokens {old_grid}->{new_grid})")
    except Exception as e:
        print("Warning: could not resize pos_embed:", e)

def adapt_pos_embed_from_checkpoint(state_dict, model, pos_name_candidates=("pos_embed","vit.pos_embed")):
    ckpt_key = None
    for k in list(state_dict.keys()):
        for cand in pos_name_candidates:
            if k.endswith(cand):
                ckpt_key = k
                break
        if ckpt_key:
            break

    if ckpt_key is None:
        return state_dict, False

    ckpt_pos = state_dict[ckpt_key] 
    model_pos = None
    try:
        model_pos = model.vit.pos_embed.data
    except Exception:
        if hasattr(model, "pos_embed"):
            model_pos = model.pos_embed.data

    if model_pos is None:
        return state_dict, False

    ckpt_N = ckpt_pos.shape[1]
    model_N = model_pos.shape[1]
    if ckpt_N == model_N:
        return state_dict, False
    ckpt_cls, ckpt_grid = ckpt_pos[:, :1, :], ckpt_pos[:, 1:, :] 
    old_grid = ckpt_grid.shape[1]
    old_size = int(math.sqrt(old_grid))
    new_grid = model_pos.shape[1] - 1
    new_size = int(math.sqrt(new_grid))

    if old_size * old_size != old_grid or new_size * new_size != new_grid:
        print("Warning: non-square pos-embed grids; skipping adaptation.")
        return state_dict, False

    D = ckpt_grid.shape[2]
    ckpt_grid_reshaped = ckpt_grid.reshape(1, old_grid, D).permute(0,2,1).reshape(1, D, old_size, old_size)
    ckpt_grid_resized = F.interpolate(ckpt_grid_reshaped, size=(new_size, new_size), mode='bilinear', align_corners=False)
    ckpt_grid_resized = ckpt_grid_resized.reshape(1, D, new_size*new_size).permute(0,2,1)
    new_pos = torch.cat([ckpt_cls, ckpt_grid_resized], dim=1) 
    state_dict[ckpt_key] = new_pos
    print(f"Adapted checkpoint pos_embed: {old_size}x{old_size} -> {new_size}x{new_size} (tokens {old_grid}->{new_grid})")
    return state_dict, True


c:\Users\Asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-02 20:39:07.892 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-02 20:39:07.894 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-02 20:39:08.238 
  command:

    streamlit run C:\Users\Asus\AppData\Roaming\Python\Python311\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-05-02 20:39:08.241 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-02 20:39:08.242 Thread 'MainThread': missing ScriptRunCon

In [3]:
def load_model_from_checkpoint(ckpt_path, device, img_size=IMG_SIZE):
    class ViTSegLocal(nn.Module):
        def __init__(self, num_classes=NUM_CLASSES):
            super().__init__()
            self.vit = timm.create_model("vit_tiny_patch16_224", pretrained=True, num_classes=0)
            self.embed_dim = self.vit.embed_dim
            self.vit.patch_embed = nn.Conv2d(in_channels=6, out_channels=self.embed_dim, kernel_size=16, stride=16)
            try:
                old = timm.create_model("vit_tiny_patch16_224", pretrained=True, num_classes=0).patch_embed.proj
                old_w = old.weight
                new_w = torch.cat([old_w, old_w], dim=1)
                self.vit.patch_embed.weight = nn.Parameter(new_w)
                self.vit.patch_embed.bias = nn.Parameter(old.bias.clone())
            except Exception:
                pass

            self.proj = nn.Conv2d(self.embed_dim, 128, kernel_size=1)
            self.decoder = nn.Sequential(
                nn.Conv2d(128, 64, 3, padding=1), nn.ReLU(),
                nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
                nn.Conv2d(64, 32, 3, padding=1), nn.ReLU(),
            )
            self.head = nn.Conv2d(32, num_classes, kernel_size=1)

        def extract_tokens(self, x):
            x = self.vit.patch_embed(x)
            if x.dim() == 4:
                B, C, Hf, Wf = x.shape
                x = x.flatten(2).transpose(1,2)
            B, N, D = x.shape
            cls = self.vit.cls_token.expand(B, -1, -1)
            x = torch.cat([cls, x], dim=1)
            x = x + self.vit.pos_embed[:, :x.shape[1], :]
            x = self.vit.pos_drop(x)
            for blk in self.vit.blocks:
                x = blk(x)
            x = self.vit.norm(x)
            return x[:,1:]

        def forward(self, x):
            B, C, H, W = x.shape
            tokens = self.extract_tokens(x)
            N = tokens.shape[1]
            S = int(math.sqrt(N))
            feat = tokens.transpose(1,2).reshape(B, self.embed_dim, S, S)
            feat = self.proj(feat)
            feat = self.decoder(feat)
            logits = self.head(feat)
            logits = nn.functional.interpolate(logits, size=(H, W), mode='bilinear', align_corners=False)
            return logits

    model = ViTSegLocal(num_classes=NUM_CLASSES).to(device)
    state = torch.load(ckpt_path, map_location="cpu")

In [4]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import cv2

def generate_failure_case_figure(model, val_loader, device, save_path="failure_case.png"):
    """
    Searches the validation set for a specific failure case (confusing Minor/Major)
    and generates a 3-panel figure for the research paper.
    """
    model.eval()
    
    # Define standard colors for xBD classes:
    # 0: Background/No Damage (Black/Dark Gray)
    # 1: Minor Damage (Orange)
    # 2: Major Damage (Red-Orange)
    # 3: Destroyed (Dark Red)
    cmap = ListedColormap(['#222222', '#FFA500', '#FF4500', '#C80000'])
    
    found_failure = False
    
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            
            # Forward pass
            logits = model(inputs)
            preds = logits.argmax(dim=1)
            
            # Look for a specific failure: Ground Truth has 'Major' (2), but Pred has 'Minor' (1)
            # We check a single image in the batch
            for i in range(inputs.size(0)):
                gt_mask = targets[i].cpu().numpy()
                pred_mask = preds[i].cpu().numpy()
                
                # Condition: GT contains Major damage, but prediction contains Minor damage
                # AND they overlap or are in the same general area
                if (gt_mask == 2).sum() > 50 and (pred_mask == 1).sum() > 50:
                    
                    # Extract the post-disaster image for visualization (channels 3,4,5)
                    post_img = inputs[i, 3:6, :, :].cpu().numpy().transpose(1, 2, 0)
                    
                    # Denormalize image to [0, 1] range for matplotlib
                    post_img = (post_img - post_img.min()) / (post_img.max() - post_img.min())
                    
                    # Create the 3-panel plot
                    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
                    
                    # Panel 1: Post-Disaster Image
                    axes[0].imshow(post_img)
                    axes[0].set_title("Post-Disaster Optical Imagery", fontsize=16, pad=15)
                    axes[0].axis('off')
                    
                    # Panel 2: Ground Truth
                    # vmin=0, vmax=3 ensures the colors map strictly to our 4 classes
                    im_gt = axes[1].imshow(gt_mask, cmap=cmap, vmin=0, vmax=3)
                    axes[1].set_title("Ground Truth (Major Damage)", fontsize=16, pad=15)
                    axes[1].axis('off')
                    
                    # Panel 3: Model Prediction
                    im_pred = axes[2].imshow(pred_mask, cmap=cmap, vmin=0, vmax=3)
                    axes[2].set_title("ViT Prediction (Minor Damage)", fontsize=16, pad=15)
                    axes[2].axis('off')
                    
                    # Add a shared colorbar
                    cbar = fig.colorbar(im_pred, ax=axes.ravel().tolist(), shrink=0.6, ticks=[0.375, 1.125, 1.875, 2.625])
                    cbar.ax.set_yticklabels(['Intact', 'Minor', 'Major', 'Destroyed'], fontsize=12)
                    
                    # Save the figure specifically for LaTeX
                    plt.savefig(save_path, bbox_inches='tight', dpi=300)
                    plt.show()
                    
                    print(f"Failure case successfully saved to {save_path}")
                    found_failure = True
                    break # Exit the inner loop
            
            if found_failure:
                break # Exit the dataloader loop

# --- How to run it ---
# Assuming 'model' is your loaded ViT and 'val_loader' is your validation DataLoader
# generate_failure_case_figure(model, val_loader, device, save_path="failure_case.png")

In [ ]:
generate_failure_case_figure(model, val_loader, device, save_path="failure_case.png")